# Which features discriminate actives from decoys

For every BaseFeatures pattern, compare hit rates in the 542 EGFR actives and the 2,500 DUD-E decoys. Enrichment, an odds ratio, Fisher's exact *p*-value, and a Benjamini–Hochberg *q*-value are reported. Pattern–pattern correlation is used to spot redundant definitions.

In [1]:
import pandas as pd
from rdkit import Chem
from pharmprof.data import load_egfr
from pharmprof.patterns import builtin_patterns
from pharmprof.profiler import profile
from pharmprof.stats import compare_classes, pattern_correlation
from pharmprof.io import MolRecord

data = load_egfr(subset=True)
recs = [MolRecord(str(r.mol_id), r.smiles, Chem.MolFromSmiles(r.smiles), int(r.label)) for r in data.itertuples()]
patterns = builtin_patterns("basefeatures")
table = profile(recs, patterns, mode="count", n_jobs=4, progress=False)
table["label"] = [r.label for r in recs]

stats = compare_classes(table, table["label"], patterns)
stats[["pattern_id", "family", "hits_actives", "hits_inactives",
       "hit_rate_actives", "hit_rate_inactives", "enrichment", "odds_ratio", "p_value", "q_value"]]

/workspace/venvs/pharmprof/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,pattern_id,family,hits_actives,hits_inactives,hit_rate_actives,hit_rate_inactives,enrichment,odds_ratio,p_value,q_value
0,basefeatures.PosIonizable.BasicGroup,PosIonizable,263,128,0.485240,0.0512,9.477341,17.468526,2.660749e-126,7.184023e-125
1,basefeatures.PosIonizable.PosN,PosIonizable,1,970,0.001845,0.3880,0.004755,0.002916,3.557112e-100,4.802102e-99
2,basefeatures.LumpedHydrophobe.iPropyl,LumpedHydrophobe,3,265,0.005535,0.1060,0.052218,0.046942,6.157865e-20,5.542078e-19
3,basefeatures.LumpedHydrophobe.RH6_6,LumpedHydrophobe,540,2323,0.996310,0.9292,1.072223,20.572536,2.321490e-13,1.253745e-12
4,basefeatures.Aromatic.Arom6,Aromatic,542,2352,1.000000,0.9408,1.062925,68.489904,2.321749e-13,1.253745e-12
5,basefeatures.LumpedHydrophobe.tButyl,LumpedHydrophobe,3,139,0.005535,0.0556,0.099551,0.094540,3.625172e-09,1.631327e-08
6,basefeatures.LumpedHydrophobe.Nitro2,LumpedHydrophobe,7,173,0.012915,0.0692,0.186635,0.175993,9.723071e-09,3.750327e-08
7,basefeatures.LumpedHydrophobe.RH5_5,LumpedHydrophobe,46,421,0.084871,0.1684,0.503984,0.457982,3.488628e-07,1.177412e-06
8,basefeatures.Hydrophobe.ThreeWayAttach,Hydrophobe,542,2447,1.000000,0.9788,1.021659,23.717058,3.883788e-05,1.165136e-04
9,basefeatures.Acceptor.SingleAtomAcceptor,Acceptor,541,2465,0.998155,0.9860,1.012328,7.681542,1.395762e-02,3.768556e-02


The only *practically* discriminating enriched feature is `PosIonizable.BasicGroup` (aliphatic / basic amines: the morpholine of gefitinib, a dimethylamino side chain, and so on). It is present in 48.5% of actives and 5.1% of decoys (enrichment 9.48, OR 17.5, *q* ≈ 7×10<sup>−125</sup>). Arom6, Acceptor and Donor are almost universal in both classes: the *p*-values are small because the set is large, but the enrichment is 1.01–1.07 and they will not separate a library. Charged nitrogen (`PosN`), isopropyl, *tert*-butyl and nitro are significantly *depleted* in the actives.

In [2]:
sig_up = stats[(stats["q_value"] < 0.05) & (stats["enrichment"] > 1)]
sig_down = stats[(stats["q_value"] < 0.05) & (stats["enrichment"] < 1)].sort_values("enrichment")
print("significantly enriched (q < 0.05, enrichment > 1):")
display(sig_up[["pattern_id", "family", "hit_rate_actives", "hit_rate_inactives", "enrichment", "odds_ratio", "q_value"]])
print("significantly depleted:")
display(sig_down[["pattern_id", "family", "hit_rate_actives", "hit_rate_inactives", "enrichment", "q_value"]])

significantly enriched (q < 0.05, enrichment > 1):


,pattern_id,family,hit_rate_actives,hit_rate_inactives,enrichment,odds_ratio,q_value
0,basefeatures.PosIonizable.BasicGroup,PosIonizable,0.485240,0.0512,9.477341,17.468526,7.184023e-125
3,basefeatures.LumpedHydrophobe.RH6_6,LumpedHydrophobe,0.996310,0.9292,1.072223,20.572536,1.253745e-12
4,basefeatures.Aromatic.Arom6,Aromatic,1.000000,0.9408,1.062925,68.489904,1.253745e-12
8,basefeatures.Hydrophobe.ThreeWayAttach,Hydrophobe,1.000000,0.9788,1.021659,23.717058,1.165136e-04
9,basefeatures.Acceptor.SingleAtomAcceptor,Acceptor,0.998155,0.9860,1.012328,7.681542,3.768556e-02


significantly depleted:


,pattern_id,family,hit_rate_actives,hit_rate_inactives,enrichment,q_value
1,basefeatures.PosIonizable.PosN,PosIonizable,0.001845,0.3880,0.004755,4.802102e-99
2,basefeatures.LumpedHydrophobe.iPropyl,LumpedHydrophobe,0.005535,0.1060,0.052218,5.542078e-19
5,basefeatures.LumpedHydrophobe.tButyl,LumpedHydrophobe,0.005535,0.0556,0.099551,1.631327e-08
6,basefeatures.LumpedHydrophobe.Nitro2,LumpedHydrophobe,0.012915,0.0692,0.186635,3.750327e-08
7,basefeatures.LumpedHydrophobe.RH5_5,LumpedHydrophobe,0.084871,0.1684,0.503984,1.177412e-06


Redundant definitions: Pearson correlation on counts, pairs with |*r*| ≥ 0.9.

In [3]:
corr, pairs = pattern_correlation(table, patterns, method="pearson", min_abs=0.9)
pairs

,pattern_a,pattern_b,correlation,method
0,basefeatures.NegIonizable.AcidicGroup,basefeatures.ZnBinder.ZnBinder5,1.0,pearson


`AcidicGroup` and `ZnBinder5` are perfectly correlated: the zinc-binder SMARTS is a carboxylic acid. Keep one of them if you are building a compact filter.